In [1]:
import os
import pandas as pd

from dotenv import load_dotenv

# Explicitly providing path to '.env'
from pathlib import Path  # Python 3.6+ only
# Load .env variables
_ = load_dotenv(dotenv_path=f"{Path().resolve().parents[1]}/src/.env")

# with the new api
from importnb import imports
with imports("ipynb"):
    from utils import to_timestamp, df_tangara_sensors, df_to_csv

PM2.5: 35.9, AQI: 102
PM2.5: 35.9, Measure Level: MeasureLevels.UNHEALTHY_FOR_SENSITIVE_GROUPS, Range Values: Min: 35.5, Max: 55.4
AQI: 102, Measure Level: MeasureLevels.UNHEALTHY_FOR_SENSITIVE_GROUPS, Range Values: Min: 101, Max: 150


## Tangara Sensors

In [2]:
# Start Date Time ISO 8601 Format, TZ='America/Bogota' -05:00
START_ISO8601_DATETIME=os.getenv("START_ISO8601_DATETIME", None)
start_timestamp = to_timestamp(START_ISO8601_DATETIME)
# End Date Time ISO 8601 Format, TZ='America/Bogota' -05:00
END_ISO8601_DATETIME=os.getenv("END_ISO8601_DATETIME", None)
end_timestamp = to_timestamp(os.getenv("END_ISO8601_DATETIME", None))

print(f'Since: {START_ISO8601_DATETIME} -> {start_timestamp}, Until: {END_ISO8601_DATETIME} -> {end_timestamp}')

2024-07-15 18:49:06.241 | DEBUG    | utils:to_timestamp:99 - datetime_iso8601: 2024-07-15T19:49:00-05:00, Timestamp: 1721090940000
2024-07-15 18:49:06.243 | DEBUG    | utils:to_timestamp:99 - datetime_iso8601: 2024-07-15T20:49:00-05:00, Timestamp: 1721094540000


Since: 2024-07-15T19:49:00-05:00 -> 1721090940000, Until: 2024-07-15T20:49:00-05:00 -> 1721094540000


In [3]:
# Data Frame Tangaras from InfluxDB
df_tangaras = df_tangara_sensors(start_timestamp, end_timestamp)
df_tangaras.drop_duplicates(subset=['MAC'], inplace=True)

print(f"Period of Time: Since: {START_ISO8601_DATETIME}, Until: {END_ISO8601_DATETIME}")
print(f"Total Tangara Sensors: {len(df_tangaras)}")

df_tangaras

2024-07-15 18:49:06.254 | DEBUG    | utils:query_tangaras:156 - sql_query: SELECT DISTINCT(geo) AS "geohash" FROM "fixed_stations_01" WHERE ("geo3" = 'd29') AND time >= 1721090940000ms AND time <= 1721094540000ms GROUP BY "name";
2024-07-15 18:49:06.343 | DEBUG    | utils:request_influxdb:131 - response: <Response [200]>
2024-07-15 18:49:06.525 | DEBUG    | utils:df_tangara_sensors:406 - Data Frame Tangaras Sensors: <class 'pandas.core.frame.DataFrame'>
Index: 38 entries, TANGARA_B7BE to TANGARA_06BE
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   GEOHASH      38 non-null     object
 1   MAC          38 non-null     object
 2   GEOLOCATION  38 non-null     object
 3   LATITUDE     38 non-null     object
 4   LONGITUDE    38 non-null     object
dtypes: object(5)
memory usage: 1.8+ KB



Period of Time: Since: 2024-07-15T19:49:00-05:00, Until: 2024-07-15T20:49:00-05:00
Total Tangara Sensors: 38


,GEOHASH,MAC,GEOLOCATION,LATITUDE,LONGITUDE
ID,,,,,
TANGARA_B7BE,d29e6de,D29ESP32DE0B7BE,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_C752,d29e6de,D29ESP32DE0C752,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_A682,d29e6d7,D29ESP32DE3A682,3.3968353271484375 -76.52595520019531,3.3968353271484375,-76.52595520019531
TANGARA_ADD6,d29e6de,D29ESP32DE3ADD6,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_B08A,d29e6de,D29ESP32DE3B08A,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_B9CA,d29e6de,D29ESP32DE3B9CA,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_BC5A,d29e6ds,D29ESP32DE3BC5A,3.3982086181640625 -76.52458190917969,3.3982086181640625,-76.52458190917969
TANGARA_BCC6,d29e6de,D29ESP32DE3BCC6,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_BD5E,d29e6de,D29ESP32DE3BD5E,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531


In [4]:
# Save Tangaras into CSV file
df_to_csv(df_tangaras, "tangaras.csv")

2024-07-15 18:49:06.558 | DEBUG    | utils:df_to_csv:311 - Save DataFrame: /home/sebaxtian/Workspaces/Tangara/tangara-evaluation/src/data/0_raw/tangaras.csv


## Reference

In [5]:
# Filter by Tangara Sensor Reference

# Tangara Sensor Reference
TANGARA_REFERENCE = os.getenv("TANGARA_REFERENCE", None)
print(f"Tangara Sensor Reference: {TANGARA_REFERENCE}")

# Filter by Reference
df_reference = df_tangaras[df_tangaras.index.isin([TANGARA_REFERENCE])]

df_reference

Tangara Sensor Reference: TANGARA_25BE


,GEOHASH,MAC,GEOLOCATION,LATITUDE,LONGITUDE
ID,,,,,
TANGARA_25BE,d29e6de,D29ESP32DE725BE,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531


In [6]:
# Save Tangara Reference into CSV file
df_to_csv(df_reference, "reference.csv", datafolder='2_features')

2024-07-15 18:49:06.585 | DEBUG    | utils:df_to_csv:311 - Save DataFrame: /home/sebaxtian/Workspaces/Tangara/tangara-evaluation/src/data/2_features/reference.csv


## Targets

In [7]:
# Filter by Tangara Sensor Targets

# Tangara Sensor Reference
TANGARA_TARGETS = os.getenv("TANGARA_TARGETS", None)
path_csvfile=f"{Path().resolve().parents[1]}/src/{TANGARA_TARGETS}"
df_targets = pd.read_csv(path_csvfile)
print(f"Total Tangara Sensor Targets: {len(df_targets)}")

# Filter by Reference
df_targets = df_tangaras[df_tangaras.index.isin(df_targets['TARGETS'])]

df_targets

Total Tangara Sensor Targets: 23


,GEOHASH,MAC,GEOLOCATION,LATITUDE,LONGITUDE
ID,,,,,
TANGARA_B7BE,d29e6de,D29ESP32DE0B7BE,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_C752,d29e6de,D29ESP32DE0C752,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_A682,d29e6d7,D29ESP32DE3A682,3.3968353271484375 -76.52595520019531,3.3968353271484375,-76.52595520019531
TANGARA_ADD6,d29e6de,D29ESP32DE3ADD6,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_B08A,d29e6de,D29ESP32DE3B08A,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_B9CA,d29e6de,D29ESP32DE3B9CA,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_BC5A,d29e6ds,D29ESP32DE3BC5A,3.3982086181640625 -76.52458190917969,3.3982086181640625,-76.52458190917969
TANGARA_BCC6,d29e6de,D29ESP32DE3BCC6,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531
TANGARA_BD5E,d29e6de,D29ESP32DE3BD5E,3.3982086181640625 -76.52595520019531,3.3982086181640625,-76.52595520019531


In [8]:
# Save Tangara Targets into CSV file
df_to_csv(df_targets, "targets.csv", datafolder='2_features')

2024-07-15 18:49:06.619 | DEBUG    | utils:df_to_csv:311 - Save DataFrame: /home/sebaxtian/Workspaces/Tangara/tangara-evaluation/src/data/2_features/targets.csv
